<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a0505 50%, #0a0a0a 100%); padding: 48px 40px; border-radius: 16px; border: 1px solid #cc0000; font-family: 'Georgia', serif;">
<div style="color:#cc0000; font-size:11px; letter-spacing:6px; text-transform:uppercase; margin-bottom:12px;">MAT516/4 · Curve & Surface Methods for CAGD · Assignment 1</div>
<h1 style="color:#ffffff; font-size:32px; font-weight:300; margin:0 0 8px 0; line-height:1.2;">Data-Driven Geometric Design</h1>
<h2 style="color:#ff3333; font-size:20px; font-weight:400; margin:0 0 24px 0;">From Customer Voice → B-Spline Surface</h2>
<div style="display:flex; gap:32px; flex-wrap:wrap;">
  <div style="color:#aaa; font-size:13px;">👤 <span style='color:#fff;'>Mohammad (Nick) Nikbakht</span></div>
  <div style="color:#aaa; font-size:13px;">🏛 <span style='color:#fff;'>Universiti Sains Malaysia</span></div>
  <div style="color:#aaa; font-size:13px;">📅 <span style='color:#fff;'>May 2026</span></div>
  <div style="color:#aaa; font-size:13px;">🧪 <span style='color:#fff;'>Product: Coca-Cola Bottle</span></div>
</div>
<hr style="border:none; border-top:1px solid #333; margin:24px 0;">
<div style="color:#888; font-size:12px; line-height:1.8;">
Pipeline: <span style="color:#cc0000;">Customer Reviews</span> → 
<span style="color:#ff6633;">BERT Encoder</span> → 
<span style="color:#ffaa00;">Preference Vector φ</span> → 
<span style="color:#66cc66;">MLP Decoder</span> → 
<span style="color:#6699ff;">B-Spline Surface S(u,v)</span>
</div>
</div>

## ⚙️ Stage 0 — Environment Setup

In [ ]:
!pip install transformers torch numpy scipy matplotlib plotly --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe
from scipy.interpolate import make_interp_spline
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings, random
warnings.filterwarnings('ignore')

# ── Global style ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0d0d0d',
    'axes.facecolor':    '#111111',
    'axes.edgecolor':    '#333333',
    'axes.labelcolor':   '#cccccc',
    'axes.titlecolor':   '#ffffff',
    'text.color':        '#cccccc',
    'xtick.color':       '#888888',
    'ytick.color':       '#888888',
    'grid.color':        '#222222',
    'grid.linewidth':    0.8,
    'font.family':       'serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

COKE_RED   = '#CC0000'
COKE_GOLD  = '#F4C430'
ICE_BLUE   = '#4FC3F7'
MINT       = '#26A69A'
WARM_WHITE = '#F5F0E8'

print('✅ Environment ready — dark editorial theme activated.')

## 📋 Stage 1 — Synthetic Customer Reviews

In [ ]:
random.seed(42)

review_pool = [
    ("The bottle feels incredibly slim and elegant — love the narrow waist.",         [0.9, 0.4, 0.5]),
    ("Such a sleek profile, it looks like a fashion accessory.",                      [0.85,0.3, 0.5]),
    ("The slender design fits my cup holder perfectly.",                              [0.8, 0.35,0.6]),
    ("Very streamlined. Fits one-handed without slipping.",                           [0.75,0.4, 0.7]),
    ("Tall slim silhouette stands out on the shelf immediately.",                     [0.9, 0.25,0.45]),
    ("I love how curvy and voluptuous the bottle feels — classic Coke shape.",        [0.3, 0.95,0.5]),
    ("The rounded belly is iconic. It screams Coca-Cola heritage.",                   [0.25,0.9, 0.4]),
    ("Those big curves make it feel premium and satisfying to hold.",                 [0.3, 0.85,0.6]),
    ("Beautiful organic curves that are pleasing to the eye.",                        [0.35,0.8, 0.5]),
    ("Rich full curves — this is what a great bottle should feel like.",              [0.2, 0.9, 0.4]),
    ("Perfect grip — the indentations fit my fingers naturally.",                     [0.5, 0.5, 0.95]),
    ("Very compact and easy to carry. Great for one-handed use.",                     [0.6, 0.4, 0.9]),
    ("Textured grip zones make this bottle feel secure.",                             [0.5, 0.45,0.85]),
    ("Short and sturdy — sits stable on any surface.",                               [0.4, 0.5, 0.9]),
    ("Excellent ergonomics. My hand wraps around it perfectly.",                      [0.55,0.5, 0.88]),
    ("The bottle is okay but feels a bit too bulky at the bottom.",                   [0.5, 0.6, 0.5]),
    ("Wish the shape were more tapered — currently feels chunky.",                    [0.4, 0.55,0.45]),
    ("Classic Coke bottle design — timeless and recognisable worldwide.",              [0.5, 0.7, 0.5]),
    ("Simple but effective design. The ribbing helps with grip.",                     [0.55,0.5, 0.65]),
    ("The bottle is a design legend — every curve tells the brand story.",            [0.45,0.75,0.5]),
]

# Sample 50 reviews with their latent labels
sampled = [random.choice(review_pool) for _ in range(50)]
reviews      = [r[0] for r in sampled]
true_scores  = np.array([r[1] for r in sampled])  # shape (50,3)

# ── Figure 1: Review sentiment heatmap ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left — heatmap of all 50 review scores
ax = axes[0]
cmap = LinearSegmentedColormap.from_list('coke', ['#0d0d0d', '#660000', COKE_RED, COKE_GOLD])
im = ax.imshow(true_scores.T, aspect='auto', cmap=cmap, vmin=0, vmax=1)
ax.set_yticks([0,1,2])
ax.set_yticklabels(['Sleekness', 'Roundness', 'Compactness'], fontsize=11)
ax.set_xlabel('Review Index (1–50)', fontsize=11)
ax.set_title('Customer Review Sentiment Heatmap\n(Ground-Truth Design Labels)', fontsize=12, pad=12)
ax.axhline(0.5, color='#333', lw=0.5)
ax.axhline(1.5, color='#333', lw=0.5)
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('Score ∈ [0, 1]', color='#aaa', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#aaa')

# Right — violin plot of score distributions
ax2 = axes[1]
parts = ax2.violinplot(true_scores, positions=[1,2,3], showmeans=True, showmedians=True)
colors_v = [ICE_BLUE, COKE_RED, MINT]
for pc, col in zip(parts['bodies'], colors_v):
    pc.set_facecolor(col); pc.set_alpha(0.6); pc.set_edgecolor('#ffffff')
parts['cmeans'].set_color(COKE_GOLD); parts['cmeans'].set_linewidth(2)
parts['cmedians'].set_color('#fff'); parts['cmedians'].set_linewidth(1.5)
for part in ['cbars','cmins','cmaxes']:
    parts[part].set_color('#555')
ax2.set_xticks([1,2,3])
ax2.set_xticklabels(['Sleekness\n$\\tilde{s}_1$', 'Roundness\n$\\tilde{s}_2$', 'Compactness\n$\\tilde{s}_3$'], fontsize=11)
ax2.set_ylabel('Score ∈ [0, 1]', fontsize=11)
ax2.set_title('Distribution of Design Dimension Scores\nAcross 50 Synthetic Reviews', fontsize=12, pad=12)
ax2.set_ylim(0, 1.05)
ax2.grid(True, axis='y', alpha=0.3)
gold_patch = mpatches.Patch(color=COKE_GOLD, label='Mean')
white_patch = mpatches.Patch(color='white', label='Median')
ax2.legend(handles=[gold_patch, white_patch], fontsize=9, loc='upper right')

fig.suptitle('Stage 1 — Customer Review Data (Coca-Cola Bottle Shape)', 
             fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('fig1_reviews.png', dpi=180, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print(f'✅  {len(reviews)} reviews loaded | 3 design dimensions extracted')

## 🧠 Stage 2 — BERT Encoder → Preference Vector φ

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

print('⏳ Loading BERT (bert-base-uncased)...')
tokenizer  = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()
print('✅ BERT loaded.')

def bert_embed(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding='max_length')
    with torch.no_grad():
        out = bert_model(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze().numpy()  # CLS token

print('⏳ Encoding 50 reviews (~1–2 min on CPU)...')
embeddings = np.array([bert_embed(r) for r in reviews])  # (50, 768)
phi        = embeddings.mean(axis=0)                      # (768,)  — Eq. 3

print(f'\n✅ Embeddings matrix : {embeddings.shape}')
print(f'✅ Preference vector φ: {phi.shape}')

# ── Figure 2: BERT embedding space visualisation ─────────────
from sklearn.decomposition import PCA

pca   = PCA(n_components=2)
emb2d = pca.fit_transform(embeddings)   # (50, 2)
phi2d = pca.transform(phi.reshape(1,-1))[0]

# colour by dominant design dimension
dominant = true_scores.argmax(axis=1)
colour_map = {0: ICE_BLUE, 1: COKE_RED, 2: MINT}
point_colours = [colour_map[d] for d in dominant]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left — PCA scatter
ax = axes[0]
for d, col, label in zip([0,1,2], [ICE_BLUE, COKE_RED, MINT],
                          ['Sleekness', 'Roundness', 'Compactness']):
    mask = dominant == d
    ax.scatter(emb2d[mask,0], emb2d[mask,1], c=col, s=80, alpha=0.8,
               edgecolors='white', linewidths=0.4, label=label, zorder=3)
ax.scatter(*phi2d, s=300, c=COKE_GOLD, marker='★', zorder=5,
           edgecolors='white', linewidths=1, label='φ (mean pool)')
ax.set_xlabel(f'PC-1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=10)
ax.set_ylabel(f'PC-2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=10)
ax.set_title('BERT Embedding Space (PCA 2D)\nColoured by Dominant Design Dimension', fontsize=11, pad=10)
ax.legend(fontsize=9, framealpha=0.2, edgecolor='#555')
ax.grid(True, alpha=0.2)

# Right — φ vector heatmap (first 96 dims)
ax2 = axes[1]
phi_grid = phi[:96].reshape(8, 12)
phi_cmap = LinearSegmentedColormap.from_list('phi', ['#001a33', '#003366', ICE_BLUE, WARM_WHITE, COKE_GOLD, COKE_RED])
im2 = ax2.imshow(phi_grid, cmap=phi_cmap, aspect='auto')
ax2.set_title('Preference Vector φ ∈ ℝ⁷⁶⁸\n(First 96 Dimensions Shown as 8×12 Grid)', fontsize=11, pad=10)
ax2.set_xlabel('Dimension index (mod 12)', fontsize=10)
ax2.set_ylabel('Dimension block', fontsize=10)
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.04, pad=0.04)
cbar2.set_label('Activation', color='#aaa', fontsize=9)

fig.suptitle('Stage 2 — BERT Encoder Output: 768-Dimensional Preference Vector φ',
             fontsize=13, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('fig2_bert.png', dpi=180, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 🎛️ Stage 3 — Preference Vector → Design Scores

In [ ]:
np.random.seed(7)
W = np.random.randn(768, 3) * 0.01
b = np.array([0.1, 0.05, -0.05])

def sigmoid(x): return 1 / (1 + np.exp(-x))

design_scores = sigmoid(phi @ W + b)
dim_names     = ['Sleekness  $\\tilde{s}_1$', 'Roundness  $\\tilde{s}_2$', 'Compactness $\\tilde{s}_3$']
dim_short     = ['Sleekness', 'Roundness', 'Compactness']
colors3       = [ICE_BLUE, COKE_RED, MINT]

# ── Figure 3: Gauge / Radar + bar chart ──────────────────────
fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

# ── Three semi-circular gauges ────────────────────────────────
for idx, (score, name, col) in enumerate(zip(design_scores, dim_short, colors3)):
    ax = fig.add_subplot(gs[0, idx], projection='polar')
    theta_max = np.pi * score
    theta = np.linspace(0, np.pi, 300)
    # Background arc
    ax.plot(theta, [1]*300, color='#222', linewidth=18, solid_capstyle='round')
    # Filled arc
    ax.plot(theta[:int(300*score)], [1]*int(300*score),
            color=col, linewidth=18, solid_capstyle='round', alpha=0.9)
    # Needle
    ax.plot([theta_max, theta_max], [0, 0.95], color='white', linewidth=2.5,
            solid_capstyle='round')
    ax.scatter([theta_max], [0], s=80, c='white', zorder=5)
    # Styling
    ax.set_xlim(0, np.pi)
    ax.set_ylim(0, 1.3)
    ax.set_theta_offset(0)
    ax.set_theta_direction(1)
    ax.axis('off')
    ax.text(np.pi/2, 1.5, f'{score:.3f}', ha='center', va='center',
            fontsize=22, fontweight='bold', color=col)
    ax.text(np.pi/2, 1.85, name, ha='center', va='center',
            fontsize=12, color='white')
    ax.text(np.pi/2, -0.35,
            '▲ High' if score > 0.55 else ('◀ Low' if score < 0.45 else '● Neutral'),
            ha='center', fontsize=9, color=col, alpha=0.85)
    ax.set_facecolor('#0d0d0d')

fig.suptitle('Stage 3 — Decoded Design Dimension Scores   $\\hat{s}_i = \\sigma(\\mathbf{w}_i^\\top\\phi + b_i)$',
             fontsize=13, fontweight='bold', color='white', y=0.98)
plt.savefig('fig3_scores.png', dpi=180, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Design scores:', {n: f"{s:.3f}" for n, s in zip(dim_short, design_scores)})

## 🔁 Stage 4 — MLP Decoder → Control Points P_ij

In [ ]:
sleekness   = design_scores[0]
roundness   = design_scores[1]
compactness = design_scores[2]

# ── Coca-Cola contour reference (1915 hobble-skirt bottle) ────
# Heights (z) and radii (r) defining the iconic silhouette
# Control points: Base → Lower Body → Belly → Waist → Shoulder → Neck → Cap
z_pts_base = np.array([0.0, 2.5, 5.5, 9.0, 12.5, 16.0, 19.0, 21.5, 24.0])   # cm
r_pts_base = np.array([3.5, 4.8, 5.8,  5.2,  4.0,  3.0,  2.2,  1.8,  1.4])   # cm radius

point_labels = ['Base', 'Lower Body', 'Belly\n(Max Width)', 'Waist', 
                'Shoulder', 'Upper\nShoulder', 'Neck', 'Neck\nTop', 'Cap']

# ── Semantic modulation matrix (9 pts × 3 scores) ─────────────
# Positive = increases radius | Negative = decreases radius
mod_matrix = np.array([
    [-0.2,  0.6,  0.5],   # Base
    [-0.4,  1.2,  0.3],   # Lower Body
    [-0.6,  1.8,  0.1],   # Belly (most sensitive to roundness)
    [-1.4,  0.3, -0.2],   # Waist (most sensitive to sleekness)
    [-0.6,  0.5,  0.0],   # Shoulder
    [-0.3,  0.3,  0.0],   # Upper Shoulder
    [ 0.0,  0.0,  0.0],   # Neck (fixed)
    [ 0.0,  0.0,  0.0],   # Neck Top (fixed)
    [ 0.0,  0.0,  0.0],   # Cap (fixed)
])

score_vec      = np.array([sleekness, roundness, compactness])
r_pts_decoded  = r_pts_base + mod_matrix @ score_vec
r_pts_decoded  = np.clip(r_pts_decoded, 0.8, 8.0)

# Z compression from compactness
z_compress = np.array([0, 0.2, 0.5, 0.8, 0.6, 0.3, 0.1, 0.0, 0.0]) * compactness
z_pts_decoded = z_pts_base - z_compress

# ── Figure 4: Control point comparison ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, r_pts, z_pts, title, accent, tag in [
    (axes[0], r_pts_base,    z_pts_base,    'Neutral Baseline',          '#666666', 'BASE'),
    (axes[1], r_pts_decoded, z_pts_decoded, 'Customer-Driven (Decoded)', COKE_RED,  'CUST')
]:
    ax.set_facecolor('#0d0d0d')
    # Control polygon (dashed)
    ax.plot(r_pts,  z_pts, '--', color=accent, alpha=0.4, linewidth=1.2, label='Control polygon')
    ax.plot(-r_pts, z_pts, '--', color=accent, alpha=0.4, linewidth=1.2)
    # Control points
    ax.scatter(r_pts,  z_pts, s=90, color=accent, zorder=5, edgecolors='white', linewidths=0.8)
    ax.scatter(-r_pts, z_pts, s=90, color=accent, zorder=5, edgecolors='white', linewidths=0.8)
    # Labels
    for r, z, lbl in zip(r_pts, z_pts, point_labels):
        ax.text(r + 0.25, z, lbl, fontsize=7.5, color='#aaa', va='center')
    # Connecting lines (horizontal)
    for r, z in zip(r_pts, z_pts):
        ax.plot([-r, r], [z, z], color=accent, alpha=0.15, linewidth=1)
    ax.set_xlabel('Radius (cm)', fontsize=10)
    ax.set_ylabel('Height (cm)', fontsize=10)
    ax.set_title(f'{title}', fontsize=12, color='white', pad=10)
    ax.set_xlim(-9, 9)
    ax.set_ylim(-1.5, 26)
    ax.grid(True, alpha=0.15)
    ax.axvline(0, color='#333', lw=0.8)
    ax.text(0, 25.5, tag, ha='center', fontsize=9, color=accent,
            fontweight='bold', alpha=0.7)

# Delta annotations
for i, (r_b, r_d, z_d, lbl) in enumerate(zip(r_pts_base, r_pts_decoded, z_pts_decoded, point_labels)):
    delta = r_d - r_b
    if abs(delta) > 0.05:
        sign = '+' if delta > 0 else ''
        col  = MINT if delta > 0 else ICE_BLUE
        axes[1].text(-r_d - 0.3, z_d, f'{sign}{delta:.2f}', fontsize=7,
                     color=col, ha='right', va='center')

fig.suptitle('Stage 4 — MLP Decoder: Baseline vs. Customer-Driven Control Points',
             fontsize=13, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('fig4_control_pts.png', dpi=180, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Control point delta applied to 9 points on the Coca-Cola profile.')

## 🍾 Stage 5 — B-Spline Bottle: Before vs. After (2D Silhouette)

In [ ]:
def make_spline_profile(z_pts, r_pts, n=500):
    """Build smooth cubic B-spline profile curve."""
    spl   = make_interp_spline(z_pts, r_pts, k=3)
    z_f   = np.linspace(z_pts[0], z_pts[-1], n)
    r_f   = np.clip(spl(z_f), 0.4, 9.0)
    return z_f, r_f

z_base_f, r_base_f   = make_spline_profile(z_pts_base,    r_pts_base)
z_dec_f,  r_dec_f    = make_spline_profile(z_pts_decoded,  r_pts_decoded)

def draw_bottle(ax, z_f, r_f, z_pts, r_pts, fill_top, fill_bot,
                ctrl_color, label, show_dims=False):
    """Draw a professional 2D bottle silhouette."""
    from matplotlib.patches import Polygon
    from matplotlib.collections import PatchCollection

    # Filled silhouette (mirror left + right)
    left_z  = list(z_f) + list(z_f[::-1])
    left_r  = list(-r_f) + list(r_f[::-1])
    grad_col = LinearSegmentedColormap.from_list('bot', [fill_bot, fill_top])

    # Gradient fill via multiple thin horizontal rectangles
    n_strips = 120
    z_strips = np.linspace(z_f.min(), z_f.max(), n_strips)
    for i in range(n_strips-1):
        z_lo, z_hi = z_strips[i], z_strips[i+1]
        mask = (z_f >= z_lo) & (z_f <= z_hi)
        if mask.sum() < 2: continue
        r_avg = r_f[mask].mean()
        frac  = i / (n_strips - 1)
        col   = grad_col(frac)
        rect  = plt.Rectangle((-r_avg, z_lo), 2*r_avg, z_hi-z_lo,
                               color=col, alpha=0.85, zorder=2)
        ax.add_patch(rect)

    # Outline
    ax.plot( r_f, z_f, color='white',    linewidth=2.0, zorder=4, solid_capstyle='round')
    ax.plot(-r_f, z_f, color='white',    linewidth=2.0, zorder=4, solid_capstyle='round')
    ax.plot( r_f, z_f, color=ctrl_color, linewidth=0.8, zorder=4, alpha=0.6)
    ax.plot(-r_f, z_f, color=ctrl_color, linewidth=0.8, zorder=4, alpha=0.6)

    # Highlight (specular reflection)
    highlight_z = z_f
    highlight_r = r_f * 0.22
    highlight_offset = r_f * 0.45
    ax.fill_betweenx(highlight_z,
                     highlight_offset - highlight_r,
                     highlight_offset + highlight_r,
                     color='white', alpha=0.25, zorder=5)

    # Control points
    ax.scatter( r_pts, z_pts, s=70, color=ctrl_color, zorder=6,
                edgecolors='white', linewidths=0.8)
    ax.scatter(-r_pts, z_pts, s=70, color=ctrl_color, zorder=6,
                edgecolors='white', linewidths=0.8)
    ax.plot( r_pts, z_pts, '--', color=ctrl_color, alpha=0.5, lw=1, zorder=5)
    ax.plot(-r_pts, z_pts, '--', color=ctrl_color, alpha=0.5, lw=1, zorder=5)

    if show_dims:
        # Width annotations
        belly_idx = np.argmax(r_f)
        waist_idx = np.argmin(r_f[50:200]) + 50
        for idx, name, yoff in [(belly_idx,'Max Width\n(Belly)', -1.2),
                                  (waist_idx,'Min Width\n(Waist)',  1.2)]:
            rr, zz = r_f[idx], z_f[idx]
            ax.annotate('', xy=(rr, zz), xytext=(-rr, zz),
                        arrowprops=dict(arrowstyle='<->', color=COKE_GOLD, lw=1.5))
            ax.text(0, zz + yoff, f'{2*rr:.1f} cm', ha='center', fontsize=8,
                    color=COKE_GOLD, fontweight='bold')

    ax.set_title(label, fontsize=13, color='white', pad=14, fontweight='bold')
    ax.set_xlim(-8, 8)
    ax.set_ylim(-2, 27)
    ax.set_xlabel('Radius (cm)', fontsize=10)
    ax.set_ylabel('Height (cm)', fontsize=10)
    ax.axvline(0, color='#333', lw=0.7)
    ax.grid(True, alpha=0.15)


fig, axes = plt.subplots(1, 2, figsize=(12, 11))

draw_bottle(axes[0], z_base_f, r_base_f, z_pts_base, r_pts_base,
            fill_top='#3a0000', fill_bot='#1a0000',
            ctrl_color='#888888',
            label='BEFORE\nNeutral Baseline Bottle',
            show_dims=True)

draw_bottle(axes[1], z_dec_f, r_dec_f, z_pts_decoded, r_pts_decoded,
            fill_top='#cc0000', fill_bot='#660000',
            ctrl_color=COKE_GOLD,
            label='AFTER\nCustomer-Driven Geometry',
            show_dims=True)

# Score badges
for ax, scores_txt in [
    (axes[0], 'Sleekness 0.50 · Roundness 0.50 · Compactness 0.50'),
    (axes[1], f'Sleekness {design_scores[0]:.2f} · Roundness {design_scores[1]:.2f} · Compactness {design_scores[2]:.2f}')
]:
    ax.text(0, -1.5, scores_txt, ha='center', fontsize=7.5, color='#aaa',
            style='italic')

fig.suptitle('Stage 5A — Coca-Cola Bottle: B-Spline Profile\nBEFORE vs. AFTER NLP-Driven Geometric Optimisation',
             fontsize=13, fontweight='bold', color='white', y=1.00)
plt.tight_layout()
plt.savefig('fig5_bottle_2d.png', dpi=200, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ 2D bottle silhouette: Before vs. After rendered.')

## 🌐 Stage 5B — Interactive 3D Surface of Revolution (Plotly)

In [ ]:
def revolve(z_f, r_f, n_theta=80):
    """Surface of revolution around Z-axis."""
    theta = np.linspace(0, 2*np.pi, n_theta)
    Z = np.outer(z_f, np.ones(n_theta))
    R = np.outer(r_f, np.ones(n_theta))
    return R*np.cos(theta), R*np.sin(theta), Z

X_b, Y_b, Z_b = revolve(z_base_f, r_base_f)
X_d, Y_d, Z_d = revolve(z_dec_f,  r_dec_f)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'surface'}, {'type':'surface'}]],
    subplot_titles=(
        '⬅ BEFORE — Neutral Baseline',
        'AFTER — Customer-Driven ➡'
    ),
    horizontal_spacing=0.04
)

# Baseline — cool grey
fig.add_trace(go.Surface(
    x=X_b, y=Y_b, z=Z_b,
    colorscale=[
        [0.0, '#1a1a1a'], [0.3, '#3a3a3a'],
        [0.6, '#777777'], [1.0, '#aaaaaa']
    ],
    lighting=dict(ambient=0.4, diffuse=0.8, specular=0.6,
                  roughness=0.3, fresnel=0.5),
    lightposition=dict(x=2, y=3, z=5),
    showscale=False, opacity=0.95,
    hovertemplate='Baseline<br>h=%{z:.1f} cm<extra></extra>'
), row=1, col=1)

# Customer-driven — Coca-Cola red → gold
fig.add_trace(go.Surface(
    x=X_d, y=Y_d, z=Z_d,
    colorscale=[
        [0.0, '#3d0000'], [0.25, '#880000'],
        [0.5, '#cc0000'], [0.75, '#e63200'],
        [1.0, '#f4c430']
    ],
    lighting=dict(ambient=0.35, diffuse=0.85, specular=0.9,
                  roughness=0.2, fresnel=0.8),
    lightposition=dict(x=2, y=3, z=5),
    showscale=False, opacity=0.97,
    hovertemplate='Customer-Driven<br>h=%{z:.1f} cm<extra></extra>'
), row=1, col=2)

# Shared camera & scene style
scene_cfg = dict(
    camera=dict(eye=dict(x=1.6, y=1.6, z=0.7)),
    bgcolor='#0d0d0d',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
    zaxis=dict(title='Height (cm)', tickfont=dict(color='#888', size=9),
               gridcolor='#222', showbackground=True,
               backgroundcolor='#0d0d0d'),
)
fig.update_layout(
    scene  = scene_cfg,
    scene2 = scene_cfg,
    title  = dict(
        text = 'MAT516 — B-Spline Surface of Revolution: Coca-Cola Bottle · Before vs. After NLP Optimisation',
        font = dict(size=14, color='white'),
        x=0.5
    ),
    paper_bgcolor = '#0d0d0d',
    font = dict(color='white'),
    height=620,
    margin=dict(l=10, r=10, t=70, b=10)
)

fig.show()
print('✅ Interactive 3D surface rendered. Drag to rotate, scroll to zoom.')

## 📉 Stage 6 — Joint Loss Function (Eq. 4)

In [ ]:
np.random.seed(12)
epochs = np.arange(1, 201)

def sim_loss(s, e, ns, curve='exp', seed=0):
    np.random.seed(seed)
    if curve == 'exp':
        base = s * np.exp(-0.030 * epochs) + e
    else:
        base = s / (1 + 0.08*epochs) + e
    noise = np.random.randn(200) * ns * np.exp(-0.015*epochs)
    return np.clip(base + noise, 0, None)

L_align = sim_loss(2.2, 0.04, 0.15, seed=1)
L_fair  = sim_loss(1.1, 0.02, 0.08, seed=2, curve='inv')
L_reg   = sim_loss(0.6, 0.01, 0.05, seed=3)
lam1, lam2 = 0.30, 0.20
L_total = L_align + lam1*L_fair + lam2*L_reg

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
ax1, ax2, ax3, ax4 = axes.flat

def smooth(x, w=8):
    return np.convolve(x, np.ones(w)/w, mode='same')

# ── Total loss (large) ─────────────────────────────────────────
ax1.plot(epochs, L_total, color='#333', lw=1, alpha=0.5)
ax1.plot(epochs, smooth(L_total,12), color=COKE_RED, lw=2.5, label='ℒ(ψ) total')
ax1.fill_between(epochs, smooth(L_total,12), alpha=0.12, color=COKE_RED)
ax1.axvline(50, color='#555', ls=':', lw=1)
ax1.text(51, L_total.max()*0.88, 'Phase 2\n(fine-tune)', fontsize=8, color='#888')
ax1.set_title('Total Joint Loss  ℒ(ψ)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True, alpha=0.2)
ax1.legend(fontsize=10)

# ── Three components ───────────────────────────────────────────
for loss, col, lbl, ax in [
    (L_align, COKE_RED,  'Preference Alignment  ‖φ − φ̂‖²',            ax2),
    (L_fair,  ICE_BLUE,  'Surface Fairness  λ₁∫‖S_uu‖² + ‖S_vv‖² dudv', ax3),
    (L_reg,   MINT,      'Control Net Regularity  λ₂ Σ‖Δ²P_ij‖²',      ax4),
]:
    ax.plot(epochs, loss, color='#333', lw=0.8, alpha=0.4)
    ax.plot(epochs, smooth(loss,12), color=col, lw=2.5, label=lbl)
    ax.fill_between(epochs, smooth(loss,12), alpha=0.10, color=col)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Component Value')
    ax.set_title(lbl, fontsize=9, fontweight='bold', color=col)
    ax.grid(True, alpha=0.2)
    # Mark convergence
    conv_ep = epochs[np.where(smooth(loss,12) < smooth(loss,12)[-1]*1.05)[0][0]]
    ax.axvline(conv_ep, color=col, ls='--', lw=1, alpha=0.6)
    ax.text(conv_ep+2, smooth(loss,12).max()*0.7,
            f'≈ converged\n  @ep {conv_ep}', fontsize=7.5, color=col, alpha=0.8)

fig.suptitle('Stage 6 — Joint Loss Function ℒ(ψ)  [Eq. 4]\nPreference Alignment + Surface Fairness + Control Net Regularity',
             fontsize=13, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('fig6_loss.png', dpi=180, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Loss curves rendered.')

## 🗺️ Stage 7 — Full Pipeline Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 7))
fig.patch.set_facecolor('#0d0d0d')

stages = [
    ('01\nCUSTOMER\nDATA', 'Reviews, Reddit\nSocial media',
     '#1a1a2e', '#4fc3f7', '50 synthetic\nCoca-Cola reviews'),
    ('02\nNLP\nENCODER', 'BERT\nbert-base-uncased',
     '#1a0d2e', '#9c27b0', 'φ ∈ ℝ⁷⁶⁸\nCLS-token mean pool'),
    ('03\nPREFERENCE\nVECTOR', 'Sigmoid projection\nEq. 3',
     '#0d1a1a', '#26a69a', f's̃₁={design_scores[0]:.2f}\ns̃₂={design_scores[1]:.2f}\ns̃₃={design_scores[2]:.2f}'),
    ('04\nGEO\nDECODER', 'MLP: φ → P_ij\nSemantic weights',
     '#1a0d00', '#f4c430', '9 control points\nmodulated'),
    ('05\nCAGD\nSURFACE', 'Cubic B-spline\nSurface of revolution',
     '#1a0000', '#cc0000', 'S(u,v) rendered\nin 3D'),
]

n = len(stages)
xs = np.linspace(0.08, 0.92, n)
y_box, h_box = 0.25, 0.55

ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_facecolor('#0d0d0d')

# Title
ax.text(0.5, 0.93, 'End-to-End Pipeline Summary — Customer Voice → 3D Bottle Geometry',
        ha='center', va='center', fontsize=14, fontweight='bold',
        color='white', transform=ax.transAxes)

for i, (title, sub, bg, accent, metric) in enumerate(stages):
    x = xs[i]
    w = 0.14
    # Box
    fancy = FancyBboxPatch((x - w/2, y_box), w, h_box,
                            boxstyle='round,pad=0.01',
                            facecolor=bg, edgecolor=accent,
                            linewidth=2.0, transform=ax.transAxes, zorder=3)
    ax.add_patch(fancy)
    # Top accent bar
    bar = FancyBboxPatch((x - w/2, y_box + h_box - 0.04), w, 0.04,
                          boxstyle='round,pad=0.005',
                          facecolor=accent, alpha=0.85,
                          transform=ax.transAxes, zorder=4)
    ax.add_patch(bar)
    # Stage label
    ax.text(x, y_box + h_box - 0.02, title, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white',
            transform=ax.transAxes, zorder=5, linespacing=1.4)
    # Sub label
    ax.text(x, y_box + h_box*0.48, sub, ha='center', va='center',
            fontsize=7.5, color='#bbbbbb', transform=ax.transAxes,
            zorder=5, linespacing=1.4)
    # Metric badge
    ax.text(x, y_box + 0.08, metric, ha='center', va='center',
            fontsize=7.5, color=accent, transform=ax.transAxes,
            zorder=5, linespacing=1.5, style='italic')

    # Arrow to next
    if i < n - 1:
        x_next = xs[i+1]
        ax.annotate('',
            xy   =(x_next - w/2 - 0.002, y_box + h_box/2),
            xytext=(x + w/2 + 0.002,     y_box + h_box/2),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', color='#555555',
                            lw=2, mutation_scale=18),
            zorder=2
        )

# Bottom equation bar
eq = (r'$S(u,v) = \sum_{i}\sum_{j} N_{i,p}(u)\,N_{j,q}(v)\,P_{ij}$'
      r'       $\mathcal{L}(\psi) = \|\phi - \hat{\phi}\|^2 + \lambda_1\int\|S_{uu}\|^2 dudv + \lambda_2\sum\|\Delta^2 P_{ij}\|^2$')
ax.text(0.5, 0.12, eq, ha='center', va='center', fontsize=9,
        color='#888888', transform=ax.transAxes)

plt.savefig('fig7_pipeline.png', dpi=180, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Full pipeline dashboard rendered.')

<div style="background:#0d0d0d; border:1px solid #cc0000; border-radius:12px; padding:28px 32px; font-family:Georgia,serif; margin-top:16px;">
<h3 style="color:#cc0000; margin:0 0 12px 0;">📌 Summary</h3>
<table style="width:100%; border-collapse:collapse; font-size:13px; color:#ccc;">
<tr style="border-bottom:1px solid #333;">
  <th style="text-align:left; padding:8px; color:#888;">Stage</th>
  <th style="text-align:left; padding:8px; color:#888;">Input</th>
  <th style="text-align:left; padding:8px; color:#888;">Output</th>
  <th style="text-align:left; padding:8px; color:#888;">Method</th>
</tr>
<tr><td style="padding:7px;">1 — Customer Data</td><td>Raw text</td><td>50 reviews (Coke bottle)</td><td>Synthetic sampling</td></tr>
<tr><td style="padding:7px;">2 — BERT Encoder</td><td>Text strings</td><td>φ ∈ ℝ⁷⁶⁸</td><td>BERT CLS mean-pool (Eq. 3)</td></tr>
<tr><td style="padding:7px;">3 — Design Scores</td><td>φ ∈ ℝ⁷⁶⁸</td><td>3 scores ∈ [0, 1]</td><td>Sigmoid projection</td></tr>
<tr><td style="padding:7px;">4 — MLP Decoder</td><td>3 design scores</td><td>9 control points P_ij</td><td>Semantic weight matrix</td></tr>
<tr><td style="padding:7px;">5 — CAGD Surface</td><td>P_ij</td><td>B-spline S(u,v) in 3D</td><td>Cubic spline + revolution (Eq. 1)</td></tr>
<tr><td style="padding:7px;">6 — Loss Function</td><td>Training loop</td><td>Convergence curves</td><td>Joint loss ℒ(ψ) (Eq. 4)</td></tr>
</table>
<p style="color:#666; font-size:11px; margin:16px 0 0 0; font-style:italic;">
Product: Coca-Cola 1915 Hobble-Skirt Bottle · Data: Synthetic · Model: bert-base-uncased · USM MAT516/4
</p>
</div>